# Burmese To English Translator

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [1]:
import torch
if torch.cuda.is_available():
    print(f"{torch.cuda.device_count()} GPU(s) detected! Device ID: {torch.cuda.current_device()}")
    device = 0
else:
    print(f"No GPU detected! Device ID: {torch.cuda.current_device()}")
    device = -1

1 GPU(s) detected! Device ID: 0


In [2]:
import torch

print(torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

NVIDIA GeForce RTX 4060 Laptop GPU
BF16 supported: True


## Load Dataset

In [3]:
from datasets import load_dataset

data = load_dataset("kalixlouiis/Myanmar-English-general-text-translation")
indices = list(range(1000))

c:\Users\User\.virtualenvs\burmese2EngTranslator-DSt_KzEO\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
data['train'][0]

{'my': 'မြို့နေလူထုသည် မန္တလေးတောင်နှင့် ကျုံးဘေးများတွင် လမ်းလျှောက်ခြင်း၊ စက်ဘီးစီးခြင်း၊ ကိုယ်လက်လှုပ်ရှားအားကစားနည်းများ ပြုလုပ်လေ့ရှိကြသည်။',
 'en': 'City residents often walk, cycle, and engage in physical exercise around Mandalay Hill and the moat.'}

## Preprocessing Data

In [5]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
model_checkpoint = "Ko-Yin-Maung/mig-burmese-llm"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [6]:
def tokenize(examples):
    prefix = "<start_of_turn>user\nTranslate to English: "
    suffix = "\n<end_of_turn><start_of_turn>model\n"
    prompt = prefix + examples["my"] + suffix + examples["en"] + "<end_of_turn>\n"
    return tokenizer(
        prompt, 
        truncation=True,
        max_length=256,
        padding=False
    )

tokenized_train_data = data['train'].map(tokenize)
tokenized_val_data = data['validation'].map(tokenize)
tokenized_test_data = data['test'].map(tokenize)

tokenized_train_data = tokenized_train_data.remove_columns(["en", "my"])
tokenized_val_data = tokenized_val_data.remove_columns(["en", "my"])
tokenized_test_data = tokenized_test_data.remove_columns(["en", "my"])

Map: 100%|██████████| 1203/1203 [00:00<00:00, 6921.58 examples/s]


In [7]:
from torch.utils.data import Subset

indices = list(range(1000))
tokenized_train_data = Subset(tokenized_train_data, list(range(100)))
tokenized_val_data = Subset(tokenized_val_data, list(range(10)))
tokenized_test_data = Subset(tokenized_test_data, list(range(10)))

In [8]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Fine-tune Model

In [9]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained(
   model_checkpoint, device_map="cuda:0",
   dtype=torch.bfloat16,
)

W0917 20:47:51.028000 25664 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 340/340 [00:02<00:00, 160.15it/s]


In [10]:
batch = data_collator([
    tokenized_val_data[i]
    for i in range(8)
])

batch = {
    k: v.to(model.device)
    for k, v in batch.items()
}

model.eval()

with torch.no_grad():
    outputs = model(**batch)

print("Loss:", outputs.loss.item())
print("Loss finite:", torch.isfinite(outputs.loss).item())
print(
    "Logits finite:",
    torch.isfinite(outputs.logits).all().item()
)
print("Logits min:", outputs.logits.min().item())
print("Logits max:", outputs.logits.max().item())

Loss: 7.775452136993408
Loss finite: True
Logits finite: True
Logits min: -37.0
Logits max: 82.5


In [11]:
training_args = TrainingArguments(
    output_dir="burmese-english-model",
    eval_strategy="epoch",
    learning_rate=3e-5,
    weight_decay=0.01,
    push_to_hub=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    logging_strategy="steps",
    logging_steps=10,
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_data,
    eval_dataset=tokenized_val_data,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,2.455771,2.409145
2,1.223168,2.568768
3,0.467811,2.949550


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.87s/it]


TrainOutput(global_step=300, training_loss=1.5574335972468059, metrics={'train_runtime': 327.669, 'train_samples_per_second': 0.916, 'train_steps_per_second': 0.916, 'total_flos': 68903278398720.0, 'train_loss': 1.5574335972468059, 'epoch': 3.0})

In [13]:
query = "ဒီလကုန်ကျရင် စာမေးပွဲရှိတယ်"
prompt = f"<start_of_turn>user\nTranslate to English: {query}\n<end_of_turn><start_of_turn>model\n"
inputs = tokenizer(
    prompt,
    return_tensors="pt",
)
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
Translate to English: ဒီလကုန်ကျရင် စာမေးပွဲရှိတယ်
model
It is the exam at the end of the term.


In [14]:
trainer.save_model("./.model")

Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.53s/it]
